In [1]:
import os, pandas as pd, random

In [2]:
cust_path = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\dataset\portfolio_customers.csv"
port_path = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\dataset\portfolio_data.csv"

cust_data = pd.read_csv(cust_path)
port_data = pd.read_csv(port_path)

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# PILLAR 1 — GOAL
# Read one customer and state clearly what they want.
# ══════════════════════════════════════════════════════════════════════════════
def goal(customer_id):
    c = cust_data[cust_data.customer_id == customer_id]
    return c   # pass to next pillar

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# PILLAR 2 — PERCEPTION
# Scan portfolio, score each row for fit, show top matches.
# ══════════════════════════════════════════════════════════════════════════════
def perception(customer):
        
    EC = customer["excluded_sectors"].values
    if "," in EC:
        excluded = [s.strip() for s in EC.split(",") if s.strip()]
    else:
        excluded = EC
    
    # preferred = [s.strip() for s in customer["preferred_sectors"].split(",") if s.strip()]
    PRF = customer["preferred_sectors"].values
    if "," in PRF:
        preferred = [s.strip() for s in PRF.split(",") if s.strip()]
    else:
        preferred = PRF

    goal_type = customer["goal_type"].values
    risk      = customer["risk_tolerance"].values
    target_ret = customer["target_annual_return_pct"].values
    esg_pref  = customer["esg_preference"].values

    
    scored = []
    # for row in port_data:
    for ndx, row in port_data.iterrows():
        # Hard filter — skip excluded sectors
        if row["sector"] in excluded:
            continue

        score = 0

        # Goal & risk alignment
        if row["goal_type"]      == goal_type: score += 3
        if row["risk_tolerance"] == risk:      score += 3

        # Sector preference
        if row["sector"] in preferred:         score += 2

        # ESG fit
        esg = row["esg_total_score"]
        if esg_pref == "Strong Preference" and esg >= 70: score += 3
        if esg_pref == "Mild Preference"   and esg >= 55: score += 1

        # Return meets target
        if row["return_1y_pct"] >= target_ret:         score += 2

    #     # Quality signals
        if row["sharpe_ratio"]  > 1.0:  score += 2
        if row["alpha_pct"]     > 2.0:  score += 2
        if row["rsi_14"]        < 40:   score += 1   # oversold
        if row["analyst_rating"] in ["Strong Buy", "Buy"]: score += 2
        if row["news_sentiment_score"] > 0.2:           score += 1

        scored.append((score, row))

    # # Sort and take top 5
    top5 = sorted(scored, key=lambda x: -x[0])[:5]

    # print("=" * 55)
    # print("  PILLAR 2 — PERCEPTION")
    # print("=" * 55)
    # print()
    # print(f"  {'#':<3} {'Ticker':<7} {'Sector':<22} {'Score':<7} "
    #       f"{'Sharpe':<8} {'1Y Ret%':<9} {'Analyst':<13} {'Rec'}")
    # print("  " + "-" * 80)

    # for rank, (sc, row) in enumerate(top5, 1):
    #     print(f"  {rank:<3} {row['ticker']:<7} {row['sector']:<22} {sc:<7} "
    #           f"{row['sharpe_ratio']:<8.2f} {row['return_1y_pct']:<9.1f} "
    #           f"{row['analyst_rating']:<13} {row['recommendation']}")
    
    return (top5)

In [5]:
random_customer = cust_data.sample(1)
customer_id = random_customer.customer_id.tolist()[0]
print(customer_id)

CUST-0056


In [6]:
c = goal(customer_id)
print(c)

   customer_id first_name last_name  age date_of_birth  gender marital_status  \
55   CUST-0056     Olivia    Wilson   72    19-06-1952  Female        Married   

    dependents       city country  ... esg_preference  \
55           3  Stockholm  Sweden  ...        Against   

                preferred_sectors excluded_sectors international_exposure  \
55  Utilities, Financials, Energy              NaN                    Yes   

   alternative_assets crypto_exposure  dividend_preference  suitability_score  \
55                Yes           Maybe        No preference                 45   

    suitability_label  goal_risk_aligned  
55             Medium               True  

[1 rows x 50 columns]


In [7]:
percep = perception(c)
print(percep)

[(16, row_id                                                                     274
date                                                                12-11-2022
portfolio_id                                                         PORT-4882
ticker                                                                     DUK
sector                                                               Utilities
                                                   ...                        
recommendation                                                      STRONG BUY
recommendation_confidence                                                 0.98
recommendation_score                                                        10
recommendation_reasoning     high risk-adj return; benchmark outperformance...
goal_risk_mismatch_flag                                                  False
Name: 273, Length: 61, dtype: object), (14, row_id                                                                     496
da

In [8]:
df_percep = pd.DataFrame()

for p in percep:
    dd = pd.DataFrame(p[1]).T
    df_percep = pd.concat([df_percep,dd],ignore_index=True)

print(df_percep)

  row_id        date portfolio_id ticker      sector asset_class  \
0    274  12-11-2022    PORT-4882    DUK   Utilities      Equity   
1    496  31-01-2022    PORT-7181    PSX      Energy      Equity   
2    977  28-04-2023    PORT-8143    AXP  Financials      Equity   
3    131  03-06-2023    PORT-1709     CF   Materials      Equity   
4    266  27-02-2022    PORT-1773     ED   Utilities      Equity   

  investment_style     benchmark currency market_regime  ...  \
0           Growth     Dow Jones      USD      Sideways  ...   
1              ESG    NASDAQ 100      USD      Recovery  ...   
2            Value       S&P 500      EUR      Volatile  ...   
3         Momentum    MSCI World      USD      Recovery  ...   
4              ESG  Russell 2000      GBP          Bull  ...   

  news_sentiment_score earnings_surprise_pct next_earnings_days  \
0                0.622                 20.31                  2   
1                0.379                 -2.04                 54   
2    

## Components of Agentic AI — Reasoning, Plan, Act

This example uses the uploaded portfolio customer and portfolio datasets.  
The agent will:

1. **Reason** — evaluate a customer profile and shortlist suitable portfolio rows.
2. **Plan** — decide a sequence of steps based on the customer's risk and goals.
3. **Act** — generate a final investment action such as BUY / HOLD / REDUCE / REVIEW.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 3 — ACT
# The agent performs the final action based on score and recommendation.
# ══════════════════════════════════════════════════════════════════════════════

def act(shortlisted_df):
    best = shortlisted_df.iloc[0]

    if best["recommendation_score"] >= 70 and best["recommendation"] in ["BUY", "HOLD"]:
        action = "INVEST / BUY"
    elif best["recommendation_score"] >= 50:
        action = "HOLD / WATCHLIST"
    else:
        action = "MANUAL REVIEW REQUIRED"

    return {
        "selected_ticker": best["ticker"],
        "sector": best["sector"],
        "agent_score": int(best["recommendation_score"]),
        "system_recommendation": best["recommendation"],
        "final_action": action,
        "why": best["recommendation_reasoning"]
    }

In [ ]:
df_percep.columns

In [ ]:
shortlisted_df = act(df_percep)
print(shortlisted_df)

{'selected_ticker': 'DUK', 'sector': 'Utilities', 'agent_score': 10, 'system_recommendation': 'STRONG BUY', 'final_action': 'MANUAL REVIEW REQUIRED', 'why': 'high risk-adj return; benchmark outperformance; analyst upgrade; positive news flow'}


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 4 — FEEDBACK
# The agent checks whether the action satisfies customer/advisor feedback.
# If not, it revises the recommendation using the next best option.
# ══════════════════════════════════════════════════════════════════════════════

def feedback(customer, shortlisted_df, action_result, customer_feedback):
    """
    customer_feedback examples:
    {
        "accepted": False,
        "reason": "too risky"
    }
    
    Other possible reasons:
    - "low esg"
    - "return too low"
    - "prefer different sector"
    """

    accepted = customer_feedback.get("accepted", True)
    reason = customer_feedback.get("reason", "accepted")

    # If customer/advisor accepts the recommendation, no change is needed.
    if accepted:
        return {
            "feedback_status": "Accepted",
            "feedback_reason": reason,
            "revised": False,
            "final_recommendation": action_result
        }

    revised_df = shortlisted_df.copy()

    # Feedback-based adjustment rules
    if reason == "too risky":
        revised_df = revised_df[revised_df["risk_tolerance"] != "Aggressive"]

    elif reason == "low esg":
        revised_df = revised_df[revised_df["esg_total_score"] >= 70]

    elif reason == "return too low":
        revised_df = revised_df[revised_df["return_1y_pct"] >= customer["target_annual_return_pct"]]

    elif reason == "prefer different sector":
        revised_df = revised_df[revised_df["sector"] != action_result["sector"]]

    # If no better alternative is available, keep the original action but mark for review.
    if revised_df.empty:
        revised_action = action_result.copy()
        revised_action["final_action"] = "MANUAL REVIEW REQUIRED"
        revised_action["why"] = revised_action["why"] + ", feedback could not be satisfied"

        return {
            "feedback_status": "Not Accepted",
            "feedback_reason": reason,
            "revised": False,
            "final_recommendation": revised_action
        }

    # Re-run action on the revised shortlist
    revised_action = act(revised_df)

    return {
        "feedback_status": "Not Accepted",
        "feedback_reason": reason,
        "revised": True,
        "final_recommendation": revised_action
    }

In [13]:
feedback(customer=customer_id, shortlisted_df=df_percep,
         customer_feedback = {"accepted": True, "reason": "good"},
         action_result = shortlisted_df     
         )

{'feedback_status': 'Accepted',
 'feedback_reason': 'good',
 'revised': False,
 'final_recommendation': {'selected_ticker': 'DUK',
  'sector': 'Utilities',
  'agent_score': 10,
  'system_recommendation': 'STRONG BUY',
  'final_action': 'MANUAL REVIEW REQUIRED',
  'why': 'high risk-adj return; benchmark outperformance; analyst upgrade; positive news flow'}}